# 04 · Limpieza · catálogo de métricas (XM)

El catálogo es la lista de las 193 métricas que publica la API de XM. No es una
serie temporal, así que no tiene huecos ni atípicos: sus problemas son de texto
y de coherencia. Todos se encontraron mirando el archivo real:

- el mismo centinela escrito de dos formas: `"No aplica"` y `"No Aplica"`;
- la misma entidad con dos grafías: `"SubArea"` y `"Subarea"`;
- 7 unidades vacías, 13 descripciones con espacios sobrantes y 5 nombres con doble espacio;
- **una URL equivocada**: el catálogo anuncia `/list` para las métricas de listado,
  pero ese endpoint responde 404; el que funciona es `/lists`.

La limpieza la hace `limpieza.catalogo.limpiar_catalogo()`: normaliza lo que se
puede normalizar, corrige solo lo que está verificado y marca lo demás.

In [1]:
import sys, warnings
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)

DATASETS = RAIZ / "datasets"
print("raiz     :", RAIZ)
print("datasets :", DATASETS, "->", "existe" if DATASETS.exists() else "FALTA")

raiz     : C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER
datasets : C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER\datasets -> existe


In [2]:
# keep_default_na=False: sin esto pandas convierte las cadenas vacías en NaN al
# leer y el problema de los vacíos quedaría escondido antes de verlo.
cat = pd.read_csv(DATASETS / "xm" / "xm_catalogo_metricas.csv", keep_default_na=False)
print(f"{len(cat)} filas · {len(cat.columns)} columnas")
cat.head()

193 filas · 9 columnas


,MetricId,MetricName,Entity,MaxDays,Type,Url,Filter,MetricUnits,MetricDescription
0,DemaReal,Demanda Real por Sistema,Sistema,31,HourlyEntities,https://servapibi.xm.com.co/hourly,No aplica,kWh,Demanda de usuarios regulados y no regulados q...
1,DemaReal,Demanda Real por Agente,Agente,31,HourlyEntities,https://servapibi.xm.com.co/hourly,Codigo Comercializador,kWh,Demanda de usuarios regulados y no regulados q...
2,ExpoMoneda,Exportaciones Moneda por Sistema,Sistema,31,HourlyEntities,https://servapibi.xm.com.co/hourly,No aplica,COP,Transferencias de Energia desde Colombia hacia...
3,DemaCome,Demanda Comercial por Sistema,Sistema,31,HourlyEntities,https://servapibi.xm.com.co/hourly,No aplica,kWh,Considera la demanda propia de cada comerciali...
4,Gene,Generación por Sistema,Sistema,31,HourlyEntities,https://servapibi.xm.com.co/hourly,No aplica,kWh,Generacion neta de cada una de las plantas Nac...


## Qué está mal, antes de tocar nada

In [3]:
texto = cat.select_dtypes("object")
print("duplicados (MetricId, Entity):", int(cat.duplicated(["MetricId", "Entity"]).sum()))
print()
print("centinela de 'sin filtro' escrito de varias formas:")
print(cat.Filter[cat.Filter.str.casefold() == "no aplica"].value_counts().to_string())
print()
grafias = cat.Entity.groupby(cat.Entity.str.casefold()).unique()
print("entidades que solo difieren en mayúsculas:", grafias[grafias.str.len() > 1].tolist())
print()
vacios = {c: int((texto[c].str.strip() == "").sum()) for c in texto}
print("celdas vacías   :", {c: n for c, n in vacios.items() if n})
sobrantes = {c: int((texto[c] != texto[c].str.strip()).sum()) for c in texto}
print("espacios de más :", {c: n for c, n in sobrantes.items() if n})
dobles = cat[cat.MetricName.str.contains("  ", regex=False)].MetricName.tolist()
print("dobles espacios :", dobles)

duplicados (MetricId, Entity): 0

centinela de 'sin filtro' escrito de varias formas:
Filter
No aplica    107
No Aplica      7

entidades que solo difieren en mayúsculas: [array(['SubArea', 'Subarea'], dtype=object)]

celdas vacías   : {'MetricUnits': 7, 'MetricDescription': 1}
espacios de más : {'MetricDescription': 13}
dobles espacios : ['Compras en Contrato Energía  Mercado No Regulado por Sistema', 'Aportes  Energía por Sistema', 'Aportes  Energía por Rio', 'Volumen Útil  diario % por Sistema', 'Volumen Útil  diario % por Embalse']


In [4]:
print("URL anunciada por tipo de métrica:")
pd.crosstab(cat.Type, cat.Url)

URL anunciada por tipo de métrica:


Url,https://servapibi.xm.com.co/daily,https://servapibi.xm.com.co/hourly,https://servapibi.xm.com.co/list,https://servapibi.xm.com.co/monthly
Type,,,,
DailyEntities,57,0,0,0
HourlyEntities,0,115,0,0
ListsEntities,0,0,7,0
MonthlyEntities,0,0,0,14


> **Sobre `/list`.** Comprobado contra el servidor el 2026-09-10: `POST /list`
> responde **404**, `POST /lists` responde 200 con las 193 métricas. Este notebook
> no llama a la API —se ejecuta igual sin red—, así que la evidencia queda escrita
> en el registro de la limpieza.

## Limpieza

In [5]:
from limpieza.catalogo import limpiar_catalogo, resumen
limpio, registro = limpiar_catalogo(cat)
print(resumen(registro))

LIMPIEZA DEL CATALOGO DE METRICAS
Filas: 193 -> 193

[normalizar_nombres]  0 filas afectadas
  nombres de columna a snake_case
  renombradas: {'MetricId': 'metric_id', 'MetricName': 'metric_name', 'Entity': 'entity', 'MaxDays': 'max_days', 'Type': 'type', 'Url': 'url', 'Filter': 'filter', 'MetricUnits': 'metric_units', 'MetricDescription': 'metric_description'}

[limpiar_texto]  23 filas afectadas
  espacios en los extremos eliminados y espacios repetidos colapsados a uno
  cambios_por_columna: {'metric_name': 5, 'metric_description': 18}
  ejemplos_originales: {'metric_name': ["'Compras en Contrato Energía  Mercado No Regulado por Sistema'", "'Aportes  Energía por Sistema'", "'Aportes  Energía por Rio'"], 'metric_description': ["'“Es el valor a cargo de los comercializadores por concepto de restricciones después de aplicar los alivios definidos en la regulación vigente”, la regla de negocio de esta métrica sería la Resolución '", "'Consumo aproximado de combustible de los generadores 

## Qué cambió en las filas tocadas

`entity` se conserva **tal cual**: es el literal que se envía a la API en el
campo `Entity`, y no está verificado que la API acepte las dos grafías. Para
agrupar y contar está `entidad_normalizada`.

In [6]:
tocadas = limpio[
    limpio.url_corregida | (limpio.entity != limpio.entidad_normalizada)
]
tocadas[["metric_id", "entity", "entidad_normalizada", "granularidad", "url", "url_corregida"]]

,metric_id,entity,entidad_normalizada,granularidad,url,url_corregida
137,DemaNoAtenProg,Subarea,SubArea,diaria,https://servapibi.xm.com.co/daily,False
138,DemaNoAtenNoProg,Subarea,SubArea,diaria,https://servapibi.xm.com.co/daily,False
186,ListadoRecursos,Agente,Agente,listado,https://servapibi.xm.com.co/lists,True
187,ListadoRecursos,Sistema,Sistema,listado,https://servapibi.xm.com.co/lists,True
188,ListadoAgentes,Sistema,Sistema,listado,https://servapibi.xm.com.co/lists,True
189,ListadoRios,Sistema,Sistema,listado,https://servapibi.xm.com.co/lists,True
190,ListadoEmbalses,Sistema,Sistema,listado,https://servapibi.xm.com.co/lists,True
191,ListadoMetricas,Sistema,Sistema,listado,https://servapibi.xm.com.co/lists,True
192,ListadoAGPE,Agente,Agente,listado,https://servapibi.xm.com.co/lists,True


In [7]:
print("filas con filtro real:", int(limpio.tiene_filtro.sum()), "de", len(limpio))
print(limpio["filter"].value_counts().to_string())
print()
print("unidades (tras convertir los vacíos a nulo):")
print(limpio.metric_units.value_counts(dropna=False).to_string())

filas con filtro real: 79 de 193
filter
Codigo Submercado Generación    38
Codigo Agente                   17
Codigo Comercializador           9
Nombre Embalse                   9
Nombre Río                       5
Codigo Distribuidor              1

unidades (tras convertir los vacíos a nulo):
metric_units
kWh          86
COP          39
COP/kWh      30
m3            7
None          7
kW            5
%             4
MBTU          3
W/m2          2
m3/s          2
gCO2e/kWh     2
USD/kWh       1
TonN2O        1
TonCH4        1
TonCO2        1
#             1
°C            1


## Comprobaciones

In [8]:
assert len(limpio) == len(cat), "se perdieron filas"
assert not limpio.duplicated(["metric_id", "entity"]).any()
assert limpio.url_coherente.all(), "hay URLs que no apuntan a su granularidad"
assert limpio.granularidad.notna().all()
assert (limpio.max_days > 0).all()
print("OK: 193 métricas, sin duplicados, todas las URL coherentes con su granularidad")

OK: 193 métricas, sin duplicados, todas las URL coherentes con su granularidad


## Guardar

Es una tabla de referencia pequeña, así que va en CSV para poder abrirla en cualquier parte.

In [9]:
import json
SALIDA = DATASETS / "limpios"
SALIDA.mkdir(parents=True, exist_ok=True)
limpio.to_csv(SALIDA / "catalogo_metricas.csv", index=False, encoding="utf-8")
with open(SALIDA / "registro_catalogo_metricas.json", "w", encoding="utf-8") as f:
    json.dump(registro, f, ensure_ascii=False, indent=2, default=str)
print(f"{len(limpio)} filas -> {SALIDA / 'catalogo_metricas.csv'}")

193 filas -> C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER\datasets\limpios\catalogo_metricas.csv
